# 08 wazuh rule based detection

Code-only reproduction copy. Outputs and execution counts have been cleared. Configure the lab settings and provide the documented input data before running. Original research evidence remains outside this folder.


In [ ]:
# Keep all inputs and outputs inside this reproduction copy.
from pathlib import Path
import sys
import re

_here = Path.cwd().resolve()
PUBLICATION_ROOT = next((p for p in (_here, _here.parent)
    if (p / '.publication-root').is_file()), None)
if PUBLICATION_ROOT is None:
    raise FileNotFoundError('Start the kernel in this repository root or its notebooks directory.')
if str(PUBLICATION_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLICATION_ROOT))
from src.lab_config import load_lab_config
LAB_CONFIG = load_lab_config()


In [ ]:
# Examine Wazuh alert coverage for training and validation sessions

from pathlib import Path
import pandas as pd

PROJECT_ROOT = PUBLICATION_ROOT

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ML_DIR = PROJECT_ROOT / "data" / "notebook_outputs" / "ml_ready"

# Load session labels for training and validation
train_df = pd.read_csv(ML_DIR / "train_ml.csv")
val_df = pd.read_csv(ML_DIR / "validation_ml.csv")

# Load Wazuh alert counts from session-level metadata
wazuh_data = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "session_features.csv",
    usecols=[
        "SessionID",
        "WazuhAlertRecordCount",
        "WazuhDistinctAlertedEventCount"
    ]
)

for partition, data in [
    ("Training", train_df),
    ("Validation", val_df)
]:
    merged = data[
        ["SessionID", "GroundTruthLabel"]
    ].merge(
        wazuh_data,
        on="SessionID",
        how="left",
        validate="one_to_one"
    )

    print(f"\n{partition.upper()} WAZUH ALERT COVERAGE")

    print("Sessions:", len(merged))

    print("Sessions with missing alert counts:",
          merged["WazuhDistinctAlertedEventCount"].isna().sum())

    print("Sessions with at least one Wazuh alert:",
          (merged["WazuhDistinctAlertedEventCount"] > 0).sum())

    print("\nAlert counts by actual class:")

    display(
        merged.groupby("GroundTruthLabel")[
            ["WazuhAlertRecordCount",
             "WazuhDistinctAlertedEventCount"]
        ].agg(["mean", "median", "max"])
    )

    print("\nSessions with and without alerts:")

    display(
        pd.crosstab(
            merged["GroundTruthLabel"],
            merged["WazuhDistinctAlertedEventCount"] > 0,
            rownames=["Actual Label"],
            colnames=["Any Wazuh Alert"]
        )
    )

In [ ]:
# Locate Wazuh alert records and inspect available rule fields

# List available Wazuh files
wazuh_dir = PROJECT_ROOT / "data" / "wazuh"

print("Files in the Wazuh directory:")
for path in sorted(wazuh_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))

# Check whether our attributed events contain alert-rule details
for filename in [
    "wazuh_attributed_events_filtered.csv",
    "wazuh_attributed_events.csv"
]:
    path = (
        PROJECT_ROOT / "data" / "notebook_outputs" / filename
    )

    if path.exists():
        columns = pd.read_csv(path, nrows=0).columns.tolist()

        rule_columns = [
            col for col in columns
            if any(term in col.lower() for term in [
                "rule", "alert", "severity", "level", "description"
            ])
        ]

        print(f"\n{filename}")
        print("Available alert-related columns:", rule_columns)

In [ ]:
# Inspect the structure of the original Wazuh alerts

import gzip
import json

alert_dir = PROJECT_ROOT / "data" / "wazuh" / "logs" / "alerts"

alert_files = sorted(alert_dir.rglob("*.json.gz"))

print("Wazuh alert files found:", len(alert_files))

sample_alert = None

# Read one alert without loading all files into memory
for filepath in alert_files:
    with gzip.open(filepath, "rt", encoding="utf-8") as f:
        for line in f:
            try:
                alert = json.loads(line)
            except json.JSONDecodeError:
                continue

            if "rule" in alert:
                sample_alert = alert
                break

    if sample_alert is not None:
        break

if sample_alert is None:
    print("No valid Wazuh alerts found.")
else:
    print("\nTop-level fields:")
    print(list(sample_alert.keys()))

    print("\nTimestamp:")
    print(sample_alert.get("timestamp"))

    print("\nWazuh rule:")
    print(sample_alert.get("rule"))

    print("\nAgent:")
    print(sample_alert.get("agent"))

    print("\nWindows event fields:")
    win = sample_alert.get("data", {}).get("win", {})
    print(win.get("system", {}))

    print("\nOther available data fields:")
    print(list(sample_alert.get("data", {}).keys()))

In [ ]:
# Examine all custom Wazuh insider-threat rules (M01-M05)

import xml.etree.ElementTree as ET
import pandas as pd

rules_dir = PROJECT_ROOT / "data" / "wazuh" / "etc" / "rules"

rule_files = sorted(rules_dir.glob("thesis_*rules.xml"))

all_rules = []

for filepath in rule_files:
    root = ET.parse(filepath).getroot()

    for rule in root.iter("rule"):
        conditions = []

        for child in rule:
            if child.tag not in {"description", "group"}:
                name = child.get("name", "")
                value = (child.text or "").strip()

                conditions.append(
                    f"{child.tag} {name}: {value}".strip()
                )

        all_rules.append({
            "SourceFile": filepath.name,
            "RuleID": rule.get("id"),
            "Level": rule.get("level"),
            "Frequency": rule.get("frequency"),
            "Timeframe": rule.get("timeframe"),
            "Description": rule.findtext("description", "").strip(),
            "Conditions": " | ".join(conditions)
        })

custom_rules_df = pd.DataFrame(all_rules)

print("Custom rule files found:", len(rule_files))
print("Total custom rules:", len(custom_rules_df))

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", 150
):
    display(custom_rules_df)

In [ ]:
# Count custom Wazuh alerts observed during development
# Do not inspect the held-out test period

import gzip
import json
from collections import Counter

# End of the validation observation period
cutoffs = pd.read_csv(
    PROJECT_ROOT / "data/notebook_outputs/session_prediction_cutoffs.csv"
)

validation_end = pd.to_datetime(
    cutoffs.loc[
        cutoffs["SessionID"].isin(val_df["SessionID"]),
        "PredictionCutoffUTC"
    ],
    utc=True
).max()

custom_rule_ids = set(
    custom_rules_df["RuleID"].astype(str)
)

rule_counts = Counter()

for filepath in alert_files:
    with gzip.open(filepath, "rt", encoding="utf-8") as f:
        for line in f:
            try:
                alert = json.loads(line)
            except json.JSONDecodeError:
                continue

            rule_id = str(alert.get("rule", {}).get("id", ""))

            if rule_id not in custom_rule_ids:
                continue

            timestamp = pd.to_datetime(
                alert.get("timestamp"),
                utc=True,
                errors="coerce"
            )

            if pd.isna(timestamp) or timestamp > validation_end:
                continue

            rule_counts[rule_id] += 1

rule_coverage = custom_rules_df[
    ["SourceFile", "RuleID", "Level", "Description"]
].copy()

rule_coverage["AlertCount"] = (
    rule_coverage["RuleID"]
    .map(rule_counts)
    .fillna(0)
    .astype(int)
)

print("Development period ends:", validation_end)
print("\nCustom Wazuh rule activity:")

display(rule_coverage)

In [ ]:
# Identify the event identifiers available for matching Wazuh alerts

attributed_path = (
    PROJECT_ROOT
    / "data"
    / "notebook_outputs"
    / "wazuh_attributed_events_filtered.csv"
)

columns = pd.read_csv(
    attributed_path,
    nrows=0
).columns.tolist()

print("Available columns for Wazuh alert attribution:")

for col in columns:
    if any(term in col.lower() for term in [
        "session", "computer", "record",
        "channel", "eventid", "timestamp",
        "time", "cutoff"
    ]):
        print(col)

In [ ]:
# Find the source of M03 and M04 Wazuh alerts

for target_rule in ["100301", "100401"]:
    found = False

    for filepath in alert_files:
        with gzip.open(filepath, "rt", encoding="utf-8") as f:
            for line in f:
                try:
                    alert = json.loads(line)
                except json.JSONDecodeError:
                    continue

                if str(alert.get("rule", {}).get("id")) != target_rule:
                    continue

                print(f"\nRULE {target_rule}")
                print("Location:", alert.get("location"))
                print("Decoder:", alert.get("decoder"))
                print("Agent:", alert.get("agent"))
                print("Timestamp:", alert.get("timestamp"))

                print("\nEvent data:")
                print(json.dumps(alert.get("data", {}), indent=2))

                found = True
                break

        if found:
            break

    if not found:
        print(f"No alert found for rule {target_rule}")

In [ ]:
# M01: Inspect custom rules and corresponding Windows events

import pandas as pd

# 1. Read the actual M01 Wazuh rules
m01_rules_path = (
    PROJECT_ROOT
    / "data/wazuh/etc/rules/thesis_insider_rules.xml"
)

print("M01 CUSTOM WAZUH RULES\n")
print(m01_rules_path.read_text())


# 2. Load our previously attributed Windows events
events = pd.read_csv(
    PROJECT_ROOT
    / "data/notebook_outputs/wazuh_attributed_events_filtered.csv",
    low_memory=False
)

# 3. Restrict to training and validation sessions
dev_ids = set(train_df["SessionID"]) | set(val_df["SessionID"])

dev_events = events[
    events["SessionID"].isin(dev_ids)
].copy()


# 4. Inspect Windows Event ID 5145 (SMB access checks)
m01_events = dev_events[
    pd.to_numeric(
        dev_events["EventID"], errors="coerce"
    ).eq(5145)
].copy()

print("\nM01 RELEVANT WINDOWS EVENTS")
print("Event 5145 records:", len(m01_events))
print("Sessions containing Event 5145:",
      m01_events["SessionID"].nunique())

print("\nSample event records:")
display(m01_events.head(3).T)

In [ ]:
# M01: Identify failed SMB access checks

print("Event 5145 audit outcomes:")
print(m01_events["SeverityValue"].value_counts(dropna=False))

# Reconstruct Wazuh rule 100100:
# Event ID 5145 AND an audit failure
m01_denials = m01_events[
    m01_events["SeverityValue"]
    .astype("string")
    .str.strip()
    .str.upper()
    .eq("AUDIT_FAILURE")
].copy()

# Load prediction cutoffs
cutoff_df = pd.read_csv(
    PROJECT_ROOT
    / "data/notebook_outputs/session_prediction_cutoffs.csv",
    usecols=["SessionID", "PredictionCutoffUTC"]
)

m01_denials = m01_denials.merge(
    cutoff_df,
    on="SessionID",
    how="left",
    validate="many_to_one"
)

# Only use evidence available by the session's cutoff
# Parse ISO 8601 timestamps with or without fractional seconds

m01_denials["AvailableTimeUTC"] = pd.to_datetime(
    m01_denials["AvailableTimeUTC"],
    format="ISO8601",
    utc=True
)

m01_denials["PredictionCutoffUTC"] = pd.to_datetime(
    m01_denials["PredictionCutoffUTC"],
    format="ISO8601",
    utc=True
)

m01_denials = m01_denials[
    m01_denials["AvailableTimeUTC"]
    <= m01_denials["PredictionCutoffUTC"]
].copy()

print("\nM01 access-denied events:", len(m01_denials))
print("Sessions with access denials:",
      m01_denials["SessionID"].nunique())

print("\nSample denied access events:")
display(
    m01_denials[
        [
            "SessionID",
            "ResolvedUsername",
            "SeverityValue",
            "ShareName",
            "EventTimeUTC",
            "AvailableTimeUTC"
        ]
    ].head(10)
)

In [ ]:
# M01: Reconstruct rule 100101
# Three SMB access denials by the same user within 60 seconds

from collections import defaultdict, deque

# Sort events chronologically by availability
m01_denials = m01_denials.copy()

m01_denials["UserKey"] = (
    m01_denials["ResolvedUsername"]
    .astype("string")
    .str.strip()
    .str.lower()
)

m01_denials = m01_denials.sort_values(
    "AvailableTimeUTC"
)

# Track recent denials for each user
user_windows = defaultdict(deque)
repeated_denial_indices = []

window_duration = pd.Timedelta(seconds=60)

for idx, event in m01_denials.iterrows():

    username = event["UserKey"]
    timestamp = event["AvailableTimeUTC"]

    if pd.isna(username) or pd.isna(timestamp):
        continue

    recent = user_windows[username]

    # Remove events outside the 60-second window
    while recent and timestamp - recent[0] > window_duration:
        recent.popleft()

    recent.append(timestamp)

    # Rule 100101: at least 3 denials within 60 seconds
    if len(recent) >= 3:
        repeated_denial_indices.append(idx)

m01_repeated = m01_denials.loc[
    repeated_denial_indices
].copy()

print("M01 RULE 100100")
print("Access-denied events:", len(m01_denials))
print("Sessions detected:", m01_denials["SessionID"].nunique())

print("\nM01 RULE 100101")
print("Events meeting repeated-denial condition:", len(m01_repeated))
print("Sessions detected:", m01_repeated["SessionID"].nunique())

print("\nSample repeated-denial detections:")
display(
    m01_repeated[
        [
            "SessionID",
            "ResolvedUsername",
            "AvailableTimeUTC",
            "ShareName"
        ]
    ].head(10)
)

In [ ]:
# M01: Combine rules 100100 and 100101 at session level

# Create the development session dataset
dev_sessions = pd.concat([
    train_df[["SessionID", "GroundTruthLabel"]].assign(
        Partition="train"
    ),
    val_df[["SessionID", "GroundTruthLabel"]].assign(
        Partition="validation"
    )
], ignore_index=True)

m01_results = dev_sessions.copy()

# Rule 100100: At least one SMB access denial
m01_results["Rule_100100"] = (
    m01_results["SessionID"]
    .isin(m01_denials["SessionID"])
    .astype(int)
)

# Rule 100101: Repeated SMB access denials
m01_results["Rule_100101"] = (
    m01_results["SessionID"]
    .isin(m01_repeated["SessionID"])
    .astype(int)
)

# Overall M01 detection: Either rule triggered
m01_results["M01_Detected"] = (
    m01_results[["Rule_100100", "Rule_100101"]]
    .max(axis=1)
)

print("M01 SESSION-LEVEL DETECTION RESULTS")

print("\nTotal sessions:", len(m01_results))
print("Sessions flagged by rule 100100:",
      m01_results["Rule_100100"].sum())
print("Sessions flagged by rule 100101:",
      m01_results["Rule_100101"].sum())
print("Sessions flagged by either rule:",
      m01_results["M01_Detected"].sum())

print("\nDetections by partition and ground-truth label:")

display(
    pd.crosstab(
        [
            m01_results["Partition"],
            m01_results["GroundTruthLabel"]
        ],
        m01_results["M01_Detected"],
        rownames=["Partition", "Actual Label"],
        colnames=["M01 Detected"]
    )
)

# Save results
wazuh_results_dir = PROJECT_ROOT / "results" / "wazuh"
wazuh_results_dir.mkdir(parents=True, exist_ok=True)

m01_results.to_csv(
    wazuh_results_dir / "m01_session_detections.csv",
    index=False
)

print("\nM01 results saved successfully.")

In [ ]:
# M02: Inspect custom rules and failed Windows login events

# 1. Read the existing M02 rules
m02_rules_path = (
    PROJECT_ROOT
    / "data/wazuh/etc/rules/thesis_m02_rules.xml"
)

print("M02 CUSTOM WAZUH RULES\n")
print(m02_rules_path.read_text(encoding="utf-8"))


# 2. Extract Windows Event ID 4625 from development sessions
# dev_events already contains training and validation events

m02_events = dev_events[
    pd.to_numeric(
        dev_events["EventID"], errors="coerce"
    ).eq(4625)
].copy()

print("\nM02 RELEVANT WINDOWS EVENTS")

print("Event 4625 records:", len(m02_events))
print(
    "Sessions containing Event 4625:",
    m02_events["SessionID"].nunique()
)


# 3. Examine the fields available for login failure detection
login_columns = [
    col for col in m02_events.columns
    if any(term in col.lower() for term in [
        "username", "status", "substatus",
        "logontype", "severity"
    ])
]

print("\nAvailable login-related columns:")
print(login_columns)


# 4. Inspect reasons for login failures
for column in ["Status", "SubStatus", "SeverityValue"]:
    if column in m02_events.columns:
        print(f"\n{column} distribution:")
        print(
            m02_events[column]
            .value_counts(dropna=False)
            .head(10)
        )


# 5. Display sample failed login events
sample_columns = [
    "SessionID",
    "ResolvedUsername",
    "TargetUsername",
    "Status",
    "SubStatus",
    "SeverityValue",
    "EventTimeUTC",
    "AvailableTimeUTC"
]

sample_columns = [
    col for col in sample_columns
    if col in m02_events.columns
]

print("\nSample failed login events:")
display(m02_events[sample_columns].head(10))

In [ ]:
# M02: Inspect the original Windows 4625 event fields

import gzip
import json

print(
    "SubStatus in processed data:",
    "SubStatus" in m02_events.columns
)

archive_dir = (
    PROJECT_ROOT
    / "data/wazuh/logs/archives/2026/Nov"
)

sample_event = None

for filepath in sorted(archive_dir.glob("*.json.gz")):

    with gzip.open(filepath, "rt", encoding="utf-8") as f:

        for line in f:
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            win = record.get("data", {}).get("win", {})
            system = win.get("system", {})

            if str(system.get("eventID")) == "4625":
                sample_event = win
                break

    if sample_event is not None:
        break

if sample_event is not None:

    eventdata = sample_event.get("eventdata", {})

    print("\nOriginal Windows event fields:")
    print(json.dumps(eventdata, indent=2))

    print("\nStatus:", eventdata.get("status"))
    print("SubStatus:", eventdata.get("subStatus"))

else:
    print("No Windows Event 4625 found.")

In [ ]:
# M02: Recover SubStatus and reconstruct rule 100200

import gzip
import json
import pandas as pd

# Match original archived events using host, channel and record ID
def event_key(computer, channel, record_id):
    return (
        str(computer).split(".")[0].strip().upper(),
        str(channel).strip().lower(),
        str(record_id).strip().removesuffix(".0")
    )

m02_events = m02_events.copy()

event_keys = [
    event_key(computer, channel, record_id)
    for computer, channel, record_id in zip(
        m02_events["Computer"],
        m02_events["EventChannel"],
        m02_events["EventRecordID"]
    )
]

target_keys = set(event_keys)
substatus_lookup = {}

archive_files = sorted(
    (PROJECT_ROOT / "data/wazuh/logs/archives")
    .rglob("ossec-archive-*.json.gz")
)

for filepath in archive_files:
    with gzip.open(filepath, "rt", encoding="utf-8") as f:
        for line in f:

            if "4625" not in line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            win = record.get("data", {}).get("win", {})
            system = win.get("system", {})

            if str(system.get("eventID")) != "4625":
                continue

            key = event_key(
                system.get("computer"),
                system.get("channel"),
                system.get("eventRecordID")
            )

            if key in target_keys:
                substatus_lookup[key] = (
                    win.get("eventdata", {}).get("subStatus")
                )

    if len(substatus_lookup) == len(target_keys):
        break

m02_events["SubStatus"] = [
    substatus_lookup.get(key)
    for key in event_keys
]

print("M02 failed-login events:", len(m02_events))
print("Events with recovered SubStatus:",
      m02_events["SubStatus"].notna().sum())

print("\nSubStatus distribution:")
print(m02_events["SubStatus"].value_counts(dropna=False))

# Rule 100200: Incorrect password
m02_incorrect_passwords = m02_events[
    m02_events["SubStatus"]
    .astype("string")
    .str.lower()
    .eq("0xc000006a")
].copy()

# Apply the original session prediction cutoffs
m02_incorrect_passwords = m02_incorrect_passwords.merge(
    cutoff_df,
    on="SessionID",
    how="left",
    validate="many_to_one"
)

for col in ["AvailableTimeUTC", "PredictionCutoffUTC"]:
    m02_incorrect_passwords[col] = pd.to_datetime(
        m02_incorrect_passwords[col],
        format="ISO8601",
        utc=True
    )

m02_incorrect_passwords = m02_incorrect_passwords[
    m02_incorrect_passwords["AvailableTimeUTC"]
    <= m02_incorrect_passwords["PredictionCutoffUTC"]
].copy()

print("\nM02 RULE 100200")
print("Timely incorrect-password events:",
      len(m02_incorrect_passwords))
print("Sessions detected:",
      m02_incorrect_passwords["SessionID"].nunique())

display(
    m02_incorrect_passwords[
        [
            "SessionID",
            "TargetUsername",
            "Status",
            "SubStatus",
            "AvailableTimeUTC"
        ]
    ].head(10)
)

In [ ]:
# M02: Reconstruct rule 100201
# Three incorrect-password attempts by the same account within 60 seconds

from collections import defaultdict, deque

m02_incorrect_passwords = m02_incorrect_passwords.copy()

# Normalise usernames for correlation
m02_incorrect_passwords["UserKey"] = (
    m02_incorrect_passwords["TargetUsername"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Process events in the order they became available
m02_incorrect_passwords = m02_incorrect_passwords.sort_values(
    "AvailableTimeUTC"
)

user_windows = defaultdict(deque)
repeated_login_indices = []

window_duration = pd.Timedelta(seconds=60)

for idx, event in m02_incorrect_passwords.iterrows():

    username = event["UserKey"]
    timestamp = event["AvailableTimeUTC"]

    if pd.isna(username) or pd.isna(timestamp):
        continue

    recent = user_windows[username]

    # Remove attempts older than 60 seconds
    while recent and timestamp - recent[0] > window_duration:
        recent.popleft()

    recent.append(timestamp)

    # Rule 100201: at least 3 incorrect passwords within 60 seconds
    if len(recent) >= 3:
        repeated_login_indices.append(idx)

m02_repeated = m02_incorrect_passwords.loc[
    repeated_login_indices
].copy()

print("M02 RULE 100200")
print("Incorrect-password events:", len(m02_incorrect_passwords))
print("Sessions detected:",
      m02_incorrect_passwords["SessionID"].nunique())

print("\nM02 RULE 100201")
print("Events meeting repeated-login condition:", len(m02_repeated))
print("Sessions detected:",
      m02_repeated["SessionID"].nunique())

print("\nSample repeated-login detections:")
display(
    m02_repeated[
        [
            "SessionID",
            "TargetUsername",
            "SubStatus",
            "AvailableTimeUTC"
        ]
    ].head(10)
)

In [ ]:
# Compare M02 scheduled sessions with actual rule detections
# Training and validation only

scenario_data = pd.read_csv(
    PROJECT_ROOT
    / "data/notebook_outputs/session_ground_truth.csv"
)

m02_review = dev_sessions.merge(
    scenario_data[["SessionID", "ScenarioID"]],
    on="SessionID",
    how="left",
    validate="one_to_one"
)

# Add detections without using labels to generate predictions
m02_review["Rule_100200"] = (
    m02_review["SessionID"]
    .isin(m02_incorrect_passwords["SessionID"])
)

m02_review["Rule_100201"] = (
    m02_review["SessionID"]
    .isin(m02_repeated["SessionID"])
)

# Examine only actual M02 sessions
m02_only = m02_review[
    m02_review["ScenarioID"] == "M02"
]

print("M02 sessions in training + validation:", len(m02_only))

print("M02 sessions detected by 100200:",
      m02_only["Rule_100200"].sum())

print("M02 sessions detected by 100201:",
      m02_only["Rule_100201"].sum())

print("\nM02 detection by partition:")
display(
    pd.crosstab(
        m02_only["Partition"],
        m02_only["Rule_100201"],
        margins=True
    )
)

print("\nAll scenarios detected by rule 100201:")
display(
    m02_review[
        m02_review["Rule_100201"]
    ]["ScenarioID"].value_counts()
)

In [ ]:
# M02: Combine rules 100200 and 100201 at session level

m02_results = dev_sessions.copy()

# Rule 100200: At least one incorrect-password attempt
m02_results["Rule_100200"] = (
    m02_results["SessionID"]
    .isin(m02_incorrect_passwords["SessionID"])
    .astype(int)
)

# Rule 100201: Repeated incorrect-password attempts
m02_results["Rule_100201"] = (
    m02_results["SessionID"]
    .isin(m02_repeated["SessionID"])
    .astype(int)
)

# Overall M02 detection: Either rule triggered
m02_results["M02_Detected"] = (
    m02_results[["Rule_100200", "Rule_100201"]]
    .max(axis=1)
)

print("M02 SESSION-LEVEL DETECTION RESULTS")

print("\nTotal sessions:", len(m02_results))

print("Sessions flagged by rule 100200:",
      m02_results["Rule_100200"].sum())

print("Sessions flagged by rule 100201:",
      m02_results["Rule_100201"].sum())

print("Sessions flagged by either rule:",
      m02_results["M02_Detected"].sum())

print("\nDetections by partition and ground-truth label:")

display(
    pd.crosstab(
        [
            m02_results["Partition"],
            m02_results["GroundTruthLabel"]
        ],
        m02_results["M02_Detected"],
        rownames=["Partition", "Actual Label"],
        colnames=["M02 Detected"]
    )
)

# Save M02 results
m02_results.to_csv(
    wazuh_results_dir / "m02_session_detections.csv",
    index=False
)

print("\nM02 results saved successfully.")

In [ ]:
# M03: Identify file-access events for retrospective detection

# Select Windows Event ID 4663
m03_events = dev_events[
    pd.to_numeric(
        dev_events["EventID"], errors="coerce"
    ).eq(4663)
].copy()

# Restrict to the Windows file server
m03_events = m03_events[
    m03_events["Computer"]
    .astype("string")
    .str.upper()
    .str.startswith(LAB_CONFIG["monitored_computer_prefix"].upper())
].copy()

# Normalise file paths
m03_events["FilePath"] = (
    m03_events["ObjectName"]
    .astype("string")
    .str.replace("/", "\\", regex=False)
    .str.replace(r"\\+", r"\\", regex=True)
)

# Only monitor files within C:\UniversityData
m03_events = m03_events[
    m03_events["FilePath"].str.contains(
        "(?i)" + re.escape(LAB_CONFIG["monitored_file_root"].rstrip("\\/") + "\\"),
        regex=True,
        na=False
    )
].copy()

print("M03 RELEVANT FILE-ACCESS EVENTS")

print("Total Event 4663 records:", len(m03_events))
print("Sessions with file-access events:",
      m03_events["SessionID"].nunique())

print("Distinct files accessed:",
      m03_events["FilePath"].nunique())

print("\nSample file-access events:")

display(
    m03_events[
        [
            "SessionID",
            "ResolvedUsername",
            "FilePath",
            "EventTimeUTC",
            "AvailableTimeUTC"
        ]
    ].head(10)
)

In [ ]:
# M03: Reconstruct five-minute file-access counts

m03_activity = m03_events.copy()

# Normalise usernames
m03_activity["UserKey"] = (
    m03_activity["ResolvedUsername"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Use Wazuh timestamps for five-minute aggregation
m03_activity["WazuhTimeUTC"] = pd.to_datetime(
    m03_activity["WazuhTimestamp"],
    format="mixed",
    utc=True,
    errors="coerce"
)

# Match the original fixed five-minute windows
m03_activity["WindowStartUTC"] = (
    m03_activity["WazuhTimeUTC"].dt.floor("5min")
)

# Count distinct paths for each user in each window
m03_window_counts = (
    m03_activity
    .dropna(subset=["UserKey", "FilePath", "WindowStartUTC"])
    .groupby(["UserKey", "WindowStartUTC"])
    .agg(
        UniqueFiles=("FilePath", "nunique"),
        EventCount=("FilePath", "size"),
        SessionCount=("SessionID", "nunique")
    )
    .reset_index()
)

# Reconstruct the threshold from thesis_m03_counter.py
m03_window_counts["Rule_100301"] = (
    m03_window_counts["UniqueFiles"] > 30
).astype(int)

# Original counter waits for the window to finish,
# followed by a 120-second ingestion delay
m03_window_counts["WindowEndUTC"] = (
    m03_window_counts["WindowStartUTC"]
    + pd.Timedelta(minutes=5)
)

m03_window_counts["EarliestRuleReadyUTC"] = (
    m03_window_counts["WindowEndUTC"]
    + pd.Timedelta(seconds=120)
)

# Display results
print("M03 RULE 100301")

print("Total five-minute windows:", len(m03_window_counts))

print(
    "Windows exceeding 30 distinct paths:",
    m03_window_counts["Rule_100301"].sum()
)

print("\nTop 15 windows by distinct path count:")

display(
    m03_window_counts
    .sort_values("UniqueFiles", ascending=False)
    .head(15)
)

In [ ]:
# M03: Associate qualifying windows with sessions
# and enforce prediction-time restrictions

# Select windows that exceeded the threshold
m03_qualifying = m03_window_counts[
    m03_window_counts["Rule_100301"] == 1
].copy()

# Identify the sessions contributing to each window
m03_window_events = m03_activity.merge(
    m03_qualifying[
        ["UserKey", "WindowStartUTC"]
    ],
    on=["UserKey", "WindowStartUTC"],
    how="inner"
)

# Prepare event availability timestamps
m03_window_events["AvailableTimeUTC"] = pd.to_datetime(
    m03_window_events["AvailableTimeUTC"],
    format="ISO8601",
    utc=True
)

# Find when at least 31 distinct paths became available
first_seen = (
    m03_window_events
    .groupby(
        ["UserKey", "WindowStartUTC", "FilePath"],
        as_index=False
    )["AvailableTimeUTC"]
    .min()
    .sort_values("AvailableTimeUTC")
)

first_seen["DistinctCount"] = (
    first_seen
    .groupby(["UserKey", "WindowStartUTC"])
    .cumcount() + 1
)

threshold_times = first_seen[
    first_seen["DistinctCount"] == 31
][
    ["UserKey", "WindowStartUTC", "AvailableTimeUTC"]
].rename(columns={
    "AvailableTimeUTC": "ThresholdReachedUTC"
})

# Identify contributing sessions
contributors = (
    m03_window_events
    .groupby(["UserKey", "WindowStartUTC"])
    .agg(
        SessionID=("SessionID", "first"),
        SessionCount=("SessionID", "nunique")
    )
    .reset_index()
)

# Drop the existing SessionCount before merging
# so we retain the count calculated from contributors

m03_candidates = (
    m03_qualifying
    .drop(columns=["SessionCount"])
    .merge(
        threshold_times,
        on=["UserKey", "WindowStartUTC"],
        validate="one_to_one"
    )
    .merge(
        contributors,
        on=["UserKey", "WindowStartUTC"],
        validate="one_to_one"
    )
)

# Do not assign windows involving multiple sessions
# to one session arbitrarily
ambiguous_count = (m03_candidates["SessionCount"] > 1).sum()

m03_candidates = m03_candidates[
    m03_candidates["SessionCount"] == 1
].copy()

# Earliest estimated detection availability:
# completed 5-minute window + 120 seconds,
# and at least 31 distinct paths observed
m03_candidates["DetectionReadyUTC"] = m03_candidates[
    ["EarliestRuleReadyUTC", "ThresholdReachedUTC"]
].max(axis=1)

# Attach prediction cutoffs
m03_candidates = m03_candidates.merge(
    cutoff_df,
    on="SessionID",
    validate="many_to_one"
)

m03_candidates["PredictionCutoffUTC"] = pd.to_datetime(
    m03_candidates["PredictionCutoffUTC"],
    format="ISO8601",
    utc=True
)

# Check whether detection was ready before prediction
m03_candidates["TimelyDetection"] = (
    m03_candidates["DetectionReadyUTC"]
    <= m03_candidates["PredictionCutoffUTC"]
)

# Produce session-level results
m03_results = dev_sessions.copy()

timely_ids = set(
    m03_candidates.loc[
        m03_candidates["TimelyDetection"],
        "SessionID"
    ]
)

m03_results["Rule_100301"] = (
    m03_results["SessionID"].isin(timely_ids).astype(int)
)

m03_results["M03_Detected"] = m03_results["Rule_100301"]

print("M03 RULE 100301")
print("Qualifying windows:", len(m03_qualifying))
print("Ambiguous multi-session windows:", ambiguous_count)
print("Unambiguous qualifying windows:", len(m03_candidates))
print(
    "Windows meeting prediction cutoff:",
    m03_candidates["TimelyDetection"].sum()
)
print(
    "Sessions detected before cutoff:",
    m03_results["M03_Detected"].sum()
)

print("\nDetections by partition and ground truth:")
display(
    pd.crosstab(
        [
            m03_results["Partition"],
            m03_results["GroundTruthLabel"]
        ],
        m03_results["M03_Detected"]
    )
)

# Save reconstructed results
m03_results.to_csv(
    wazuh_results_dir / "m03_session_detections.csv",
    index=False
)

print("\nM03 results saved successfully.")

In [ ]:
# M03: Examine eventual detections and prediction-time delays

m03_candidates["DelayAfterCutoffMinutes"] = (
    m03_candidates["DetectionReadyUTC"]
    - m03_candidates["PredictionCutoffUTC"]
).dt.total_seconds() / 60

print("M03 TIMING ANALYSIS")

print("Unambiguous qualifying windows:", len(m03_candidates))

print(
    "Distinct sessions with eventual detection:",
    m03_candidates["SessionID"].nunique()
)

print(
    "Sessions detected before cutoff:",
    m03_results["M03_Detected"].sum()
)

print("\nDetection delay relative to prediction cutoff (minutes):")

print(
    m03_candidates["DelayAfterCutoffMinutes"]
    .describe()
    .round(2)
)

# Add an eventual-detection indicator for analysis only
m03_results["M03_Eventual_Detected"] = (
    m03_results["SessionID"]
    .isin(m03_candidates["SessionID"])
    .astype(int)
)

print("\nEventual detections by partition and actual label:")

display(
    pd.crosstab(
        [
            m03_results["Partition"],
            m03_results["GroundTruthLabel"]
        ],
        m03_results["M03_Eventual_Detected"]
    )
)

print("\nSample detection times:")

display(
    m03_candidates[
        [
            "SessionID",
            "UniqueFiles",
            "DetectionReadyUTC",
            "PredictionCutoffUTC",
            "DelayAfterCutoffMinutes"
        ]
    ].sort_values("DelayAfterCutoffMinutes").head(10)
)

In [ ]:
# M03: Near-real-time detection using a rolling five-minute window

from collections import defaultdict, deque, Counter

# Use the M03 events we already prepared
m03_stream = m03_activity[
    ["SessionID", "UserKey", "FilePath", "AvailableTimeUTC"]
].copy()

# Parse event availability timestamps
m03_stream["AvailableTimeUTC"] = pd.to_datetime(
    m03_stream["AvailableTimeUTC"],
    format="ISO8601",
    utc=True,
    errors="coerce"
)

# Normalise file paths for distinct-file counting
m03_stream["FileKey"] = (
    m03_stream["FilePath"]
    .astype("string")
    .str.strip()
    .str.lower()
)

m03_stream = (
    m03_stream
    .dropna(subset=[
        "SessionID",
        "UserKey",
        "FileKey",
        "AvailableTimeUTC"
    ])
    .sort_values("AvailableTimeUTC")
)

# Maintain a rolling five-minute window for each user
user_events = defaultdict(deque)
user_file_counts = defaultdict(Counter)

realtime_detections = []

window_duration = pd.Timedelta(minutes=5)
threshold = 30

for event in m03_stream.itertuples(index=False):

    user = event.UserKey
    file_path = event.FileKey
    timestamp = event.AvailableTimeUTC

    recent = user_events[user]
    file_counts = user_file_counts[user]

    # Remove events older than five minutes
    while recent and (
        timestamp - recent[0][0] > window_duration
    ):
        old_time, old_file = recent.popleft()

        file_counts[old_file] -= 1

        if file_counts[old_file] == 0:
            del file_counts[old_file]

    # Count distinct files before the new event
    previous_count = len(file_counts)

    # Add the newly observed file-access event
    recent.append((timestamp, file_path))
    file_counts[file_path] += 1

    current_count = len(file_counts)

    # Trigger when the threshold is first crossed
    if previous_count <= threshold and current_count > threshold:

        realtime_detections.append({
            "SessionID": event.SessionID,
            "UserKey": user,
            "DetectionReadyUTC": timestamp,
            "DistinctFiles": current_count
        })

# Create detection DataFrame
m03_realtime_alerts = pd.DataFrame(
    realtime_detections,
    columns=[
        "SessionID",
        "UserKey",
        "DetectionReadyUTC",
        "DistinctFiles"
    ]
)

# Attach the original ML prediction cutoffs
m03_realtime_alerts = m03_realtime_alerts.merge(
    cutoff_df,
    on="SessionID",
    how="left",
    validate="many_to_one"
)

m03_realtime_alerts["PredictionCutoffUTC"] = pd.to_datetime(
    m03_realtime_alerts["PredictionCutoffUTC"],
    format="ISO8601",
    utc=True
)

# Check whether each alert was available before prediction
m03_realtime_alerts["TimelyDetection"] = (
    m03_realtime_alerts["DetectionReadyUTC"]
    <= m03_realtime_alerts["PredictionCutoffUTC"]
)

# Create session-level results
m03_realtime_results = dev_sessions.copy()

timely_ids = set(
    m03_realtime_alerts.loc[
        m03_realtime_alerts["TimelyDetection"],
        "SessionID"
    ]
)

m03_realtime_results["Rule_100301_Realtime"] = (
    m03_realtime_results["SessionID"]
    .isin(timely_ids)
    .astype(int)
)

# Display results
print("M03 NEAR-REAL-TIME DETECTION RESULTS")

print("Total threshold-crossing alerts:",
      len(m03_realtime_alerts))

print("Distinct sessions with threshold crossings:",
      m03_realtime_alerts["SessionID"].nunique())

print("Sessions detected before ML prediction cutoff:",
      m03_realtime_results["Rule_100301_Realtime"].sum())

# Compare against the 39 previously identified sessions
previous_eventual_ids = set(m03_candidates["SessionID"])

print(
    "Previously identified eventual sessions now detected in time:",
    len(timely_ids & previous_eventual_ids),
    "of",
    len(previous_eventual_ids)
)

print("\nDetections by partition and ground truth:")

display(
    pd.crosstab(
        [
            m03_realtime_results["Partition"],
            m03_realtime_results["GroundTruthLabel"]
        ],
        m03_realtime_results["Rule_100301_Realtime"]
    )
)

print("\nSample real-time detections:")

display(m03_realtime_alerts.head(10))

# Save separately from the original delayed M03 results
m03_realtime_results.to_csv(
    wazuh_results_dir / "m03_realtime_session_detections.csv",
    index=False
)

m03_realtime_alerts.to_csv(
    wazuh_results_dir / "m03_realtime_alerts.csv",
    index=False
)

print("\nNear-real-time M03 results saved successfully.")

In [ ]:
# Show total distinct files accessed during each detected session

session_file_counts = (
    m03_stream
    .groupby("SessionID")["FileKey"]
    .nunique()
    .rename("TotalSessionDistinctFiles")
)

m03_alert_details = m03_realtime_alerts.merge(
    session_file_counts,
    on="SessionID",
    how="left",
    validate="many_to_one"
)

display(
    m03_alert_details[
        [
            "SessionID",
            "UserKey",
            "DistinctFiles",
            "TotalSessionDistinctFiles",
            "DetectionReadyUTC",
            "TimelyDetection"
        ]
    ].head(15)
)

In [ ]:
# M04: Identify deletion-access events

# Reuse M03 events, which already contain:
# Event ID 4663, Windows file-server activity,
# and paths under C:\UniversityData

print("Access mask distribution:")
print(
    m03_activity["AccessMask"]
    .astype("string")
    .str.strip()
    .str.lower()
    .value_counts()
    .head(15)
)

# Match the original M04 counter's DELETE access condition
m04_events = m03_activity[
    m03_activity["AccessMask"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("0x10000")
].copy()

print("\nM04 RELEVANT DELETE-ACCESS EVENTS")

print("Total DELETE-access events:", len(m04_events))

print(
    "Sessions containing DELETE-access events:",
    m04_events["SessionID"].nunique()
)

print(
    "Distinct file paths:",
    m04_events["FilePath"].nunique()
)

print("\nSample deletion-access events:")

display(
    m04_events[
        [
            "SessionID",
            "ResolvedUsername",
            "FilePath",
            "AccessMask",
            "EventTimeUTC",
            "AvailableTimeUTC"
        ]
    ].head(10)
)

In [ ]:
# M04: Near-real-time mass deletion detection
# Rule 100401: At least 5 distinct paths with DELETE access
# within a rolling five-minute window

from collections import defaultdict, deque, Counter

# Prepare M04 events
m04_stream = m04_events[
    ["SessionID", "UserKey", "FilePath", "AvailableTimeUTC"]
].copy()

m04_stream["AvailableTimeUTC"] = pd.to_datetime(
    m04_stream["AvailableTimeUTC"],
    format="ISO8601",
    utc=True,
    errors="coerce"
)

m04_stream["FileKey"] = (
    m04_stream["FilePath"]
    .astype("string")
    .str.strip()
    .str.lower()
)

m04_stream = (
    m04_stream
    .dropna(subset=[
        "SessionID",
        "UserKey",
        "FileKey",
        "AvailableTimeUTC"
    ])
    .sort_values("AvailableTimeUTC")
)

# Rolling five-minute detection
user_events = defaultdict(deque)
user_file_counts = defaultdict(Counter)

m04_detections = []

window_duration = pd.Timedelta(minutes=5)
threshold = 5

for event in m04_stream.itertuples(index=False):

    user = event.UserKey
    file_path = event.FileKey
    timestamp = event.AvailableTimeUTC

    recent = user_events[user]
    file_counts = user_file_counts[user]

    # Remove events outside the rolling window
    while recent and timestamp - recent[0][0] > window_duration:

        old_time, old_file = recent.popleft()

        file_counts[old_file] -= 1

        if file_counts[old_file] == 0:
            del file_counts[old_file]

    previous_count = len(file_counts)

    # Add current DELETE-access event
    recent.append((timestamp, file_path))
    file_counts[file_path] += 1

    current_count = len(file_counts)

    # Detect when the fifth distinct path is reached
    if previous_count < threshold and current_count >= threshold:

        m04_detections.append({
            "SessionID": event.SessionID,
            "UserKey": user,
            "DetectionReadyUTC": timestamp,
            "DistinctFilesAtDetection": current_count
        })

# Create alert DataFrame
m04_realtime_alerts = pd.DataFrame(
    m04_detections,
    columns=[
        "SessionID",
        "UserKey",
        "DetectionReadyUTC",
        "DistinctFilesAtDetection"
    ]
)

# Attach the original ML prediction cutoffs
m04_realtime_alerts = m04_realtime_alerts.merge(
    cutoff_df,
    on="SessionID",
    how="left",
    validate="many_to_one"
)

m04_realtime_alerts["PredictionCutoffUTC"] = pd.to_datetime(
    m04_realtime_alerts["PredictionCutoffUTC"],
    format="ISO8601",
    utc=True
)

# Only count detections available before prediction cutoff
m04_realtime_alerts["TimelyDetection"] = (
    m04_realtime_alerts["DetectionReadyUTC"]
    <= m04_realtime_alerts["PredictionCutoffUTC"]
)

# Create one detection result per session
m04_results = dev_sessions.copy()

timely_ids = set(
    m04_realtime_alerts.loc[
        m04_realtime_alerts["TimelyDetection"],
        "SessionID"
    ]
)

m04_results["Rule_100401"] = (
    m04_results["SessionID"]
    .isin(timely_ids)
    .astype(int)
)

m04_results["M04_Detected"] = m04_results["Rule_100401"]

# Display results
print("M04 NEAR-REAL-TIME DETECTION RESULTS")

print("Total threshold-crossing alerts:", len(m04_realtime_alerts))

print(
    "Distinct sessions with threshold crossings:",
    m04_realtime_alerts["SessionID"].nunique()
)

print(
    "Sessions detected before ML prediction cutoff:",
    m04_results["M04_Detected"].sum()
)

print("\nDetections by partition and ground truth:")

display(
    pd.crosstab(
        [
            m04_results["Partition"],
            m04_results["GroundTruthLabel"]
        ],
        m04_results["M04_Detected"]
    )
)

print("\nSample M04 detections:")
display(m04_realtime_alerts.head(10))

# Save results
m04_results.to_csv(
    wazuh_results_dir / "m04_realtime_session_detections.csv",
    index=False
)

m04_realtime_alerts.to_csv(
    wazuh_results_dir / "m04_realtime_alerts.csv",
    index=False
)

print("\nM04 results saved successfully.")

In [ ]:
# M05: Inspect the custom Wazuh rule and SFTP activity

# Display the M05 rule
m05_rule_path = (
    PROJECT_ROOT
    / "data/wazuh/etc/rules/thesis_m05_rules.xml"
)

print("M05 CUSTOM WAZUH RULE\n")
print(m05_rule_path.read_text(encoding="utf-8"))


# Identify Sysmon process creation events (Event ID 1)
m05_process_events = dev_events[
    pd.to_numeric(
        dev_events["EventID"], errors="coerce"
    ).eq(1)
    &
    dev_events["Provider"].astype("string").str.contains(
        "Sysmon", case=False, na=False
    )
].copy()

# Identify SFTP process activity
m05_sftp_events = m05_process_events[
    m05_process_events["Image"]
    .astype("string")
    .str.contains("sftp.exe", case=False, regex=False, na=False)
    |
    m05_process_events["CommandLine"]
    .astype("string")
    .str.contains("sftp", case=False, regex=False, na=False)
].copy()

print("\nM05 SFTP ACTIVITY")
print("Sysmon process creation events:", len(m05_process_events))
print("SFTP-related events:", len(m05_sftp_events))
print("Sessions with SFTP activity:",
      m05_sftp_events["SessionID"].nunique())

display(
    m05_sftp_events[
        [
            "SessionID",
            "ResolvedUsername",
            "Image",
            "CommandLine",
            "EventTimeUTC",
            "AvailableTimeUTC"
        ]
    ].head(10)
)

In [ ]:
# M05: Reconstruct custom Wazuh rule 100501

# Apply the exact image and destination conditions
m05_rule_matches = m05_process_events[
    m05_process_events["Image"]
    .fillna("")
    .astype(str)
    .str.contains(
        r"\\sftp\.exe$",
        case=False,
        regex=True
    )
    &
    m05_process_events["CommandLine"]
    .fillna("")
    .astype(str)
    .str.contains(
        LAB_CONFIG["m05_destination"],
        case=False,
        regex=False
    )
].copy()

print("M05 RULE 100501")
print("Matching process events:", len(m05_rule_matches))
print(
    "Distinct sessions:",
    m05_rule_matches["SessionID"].nunique()
)

# Check which scenarios produced matching events
if "ScenarioID" in m05_rule_matches.columns:
    print("\nMatching events by scenario:")
    print(m05_rule_matches["ScenarioID"].value_counts())

# Inspect a few matching records
preview = m05_rule_matches.copy()

preview["CommandLineTail"] = (
    preview["CommandLine"].astype(str).str[-150:]
)

display(
    preview[
        [
            "SessionID",
            "ResolvedUsername",
            "Image",
            "CommandLineTail",
            "EventTimeUTC",
            "AvailableTimeUTC"
        ]
    ].head(10)
)

In [ ]:
# M05: Check detection timing against ML prediction cutoff

m05_timing = m05_rule_matches.copy()

# Retrieve the established session prediction cutoffs
m05_cutoffs = cutoff_df[
    ["SessionID", "PredictionCutoffUTC"]
].drop_duplicates()

m05_timing = m05_timing.merge(
    m05_cutoffs,
    on="SessionID",
    how="left",
    validate="many_to_one"
)

# Convert timestamps, including mixed ISO formats
for col in [
    "EventTimeUTC",
    "AvailableTimeUTC",
    "PredictionCutoffUTC"
]:
    m05_timing[col] = pd.to_datetime(
        m05_timing[col],
        utc=True,
        format="mixed",
        errors="raise"
    )

if m05_timing["PredictionCutoffUTC"].isna().any():
    raise ValueError("Some M05 events have no prediction cutoff")

# Detection cannot occur before the event is generated
# or before the event becomes available to Wazuh
m05_timing["DetectionReadyUTC"] = m05_timing[
    ["EventTimeUTC", "AvailableTimeUTC"]
].max(axis=1)

m05_timing["TimelyDetection"] = (
    m05_timing["DetectionReadyUTC"]
    <= m05_timing["PredictionCutoffUTC"]
)

print("M05 RULE 100501 TIMING RESULTS")

print(
    "Total matching events:",
    len(m05_timing)
)

print(
    "Distinct matched sessions:",
    m05_timing["SessionID"].nunique()
)

print(
    "Sessions detected before ML cutoff:",
    m05_timing.loc[
        m05_timing["TimelyDetection"], "SessionID"
    ].nunique()
)

print(
    "Sessions detected after ML cutoff:",
    m05_timing.loc[
        ~m05_timing["TimelyDetection"], "SessionID"
    ].nunique()
)

print("\nSample M05 detections:")

display(
    m05_timing[
        [
            "SessionID",
            "ResolvedUsername",
            "EventTimeUTC",
            "DetectionReadyUTC",
            "PredictionCutoffUTC",
            "TimelyDetection"
        ]
    ].head(15)
)

In [ ]:
# M05: Detection latency sensitivity analysis

print("M05 DETECTION LATENCY ANALYSIS\n")

for delay_seconds in [0, 1, 2, 3, 5, 10, 30]:

    estimated_detection_time = (
        m05_timing["DetectionReadyUTC"]
        + pd.Timedelta(seconds=delay_seconds)
    )

    timely = (
        estimated_detection_time
        <= m05_timing["PredictionCutoffUTC"]
    )

    detected_sessions = m05_timing.loc[
        timely, "SessionID"
    ].nunique()

    print(
        f"Additional delay: {delay_seconds:2d}s | "
        f"Timely detections: {detected_sessions}/27"
    )

In [ ]:
# M05: Save session-level detections and latency scenarios

m05_results = dev_sessions[
    ["SessionID", "Partition", "GroundTruthLabel"]
].copy()

if m05_results["SessionID"].duplicated().any():
    raise ValueError("Duplicate session IDs in development dataset")

# Calculate detection outcomes under different delays
for delay in [0, 3, 5, 10]:

    detection_time = (
        m05_timing["DetectionReadyUTC"]
        + pd.Timedelta(seconds=delay)
    )

    timely_ids = set(
        m05_timing.loc[
            detection_time <= m05_timing["PredictionCutoffUTC"],
            "SessionID"
        ]
    )

    m05_results[f"M05_Detected_{delay}s"] = (
        m05_results["SessionID"]
        .isin(timely_ids)
        .astype(int)
    )

# Preserve the zero-delay reconstruction separately
m05_results["Rule_100501"] = m05_results["M05_Detected_0s"]

# Check results
print("M05 SESSION-LEVEL RESULTS\n")

for delay in [0, 3, 5, 10]:
    col = f"M05_Detected_{delay}s"

    print(
        f"Additional delay {delay}s: "
        f"{m05_results[col].sum()} detected sessions"
    )

print("\nZero-delay detections by partition and label:")

display(
    pd.crosstab(
        [
            m05_results["Partition"],
            m05_results["GroundTruthLabel"]
        ],
        m05_results["M05_Detected_0s"]
    )
)

# Save results
m05_results.to_csv(
    wazuh_results_dir / "m05_session_detections.csv",
    index=False
)

print("\nM05 results saved successfully.")

In [ ]:
# Combine M01-M05 into one Wazuh-only baseline

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# Start with the existing development sessions
wazuh_baseline = dev_sessions.copy()

if wazuh_baseline["SessionID"].duplicated().any():
    raise ValueError("Duplicate development SessionIDs")

def add_detection_file(filename, columns):
    global wazuh_baseline

    df = pd.read_csv(wazuh_results_dir / filename)

    if df["SessionID"].duplicated().any():
        raise ValueError(f"Duplicate SessionIDs in {filename}")

    if set(df["SessionID"]) != set(wazuh_baseline["SessionID"]):
        raise ValueError(f"Session coverage mismatch: {filename}")

    wazuh_baseline = wazuh_baseline.merge(
        df[["SessionID"] + columns],
        on="SessionID",
        how="left",
        validate="one_to_one"
    )

# M01 and M02
add_detection_file(
    "m01_session_detections.csv",
    ["M01_Detected"]
)

add_detection_file(
    "m02_session_detections.csv",
    ["M02_Detected"]
)

# M03: Use the revised rolling-window alerts
m03_alerts = pd.read_csv(
    wazuh_results_dir / "m03_realtime_alerts.csv"
)

m03_timely = set(
    m03_alerts.loc[
        m03_alerts["TimelyDetection"]
        .astype(str).str.lower().eq("true"),
        "SessionID"
    ]
)

wazuh_baseline["M03_Detected"] = (
    wazuh_baseline["SessionID"]
    .isin(m03_timely)
    .astype(int)
)

# M04: Use the revised rolling-window results
add_detection_file(
    "m04_realtime_session_detections.csv",
    ["M04_Detected"]
)

# M05: Preserve processing-delay assumptions
m05_columns = [
    "M05_Detected_0s",
    "M05_Detected_3s",
    "M05_Detected_5s",
    "M05_Detected_10s"
]

add_detection_file(
    "m05_session_detections.csv",
    m05_columns
)

# Combine detections with logical OR
common_rules = [
    "M01_Detected",
    "M02_Detected",
    "M03_Detected",
    "M04_Detected"
]

for delay in [0, 3, 5, 10]:
    output_col = f"Wazuh_Detected_M05delay_{delay}s"

    wazuh_baseline[output_col] = wazuh_baseline[
        common_rules + [f"M05_Detected_{delay}s"]
    ].max(axis=1)

# Evaluate only on the validation partition
validation = wazuh_baseline[
    wazuh_baseline["Partition"] == "validation"
].copy()

assert len(validation) == 231, "Unexpected validation size"

y_true = validation["GroundTruthLabel"].astype(int)

rows = []

for delay in [0, 3, 5, 10]:
    col = f"Wazuh_Detected_M05delay_{delay}s"
    y_pred = validation[col].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    rows.append({
        "M05_ExtraDelaySeconds": delay,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "FalsePositiveRate": (
            fp / (fp + tn) if fp + tn else 0
        ),
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn
    })

wazuh_metrics = pd.DataFrame(rows)

print("WAZUH RETROSPECTIVE VALIDATION RESULTS")
display(wazuh_metrics.round(4))

# Inspect detections by scenario
if "ScenarioID" in validation.columns:
    print("\nDETECTION BY SCENARIO (0s M05 delay)")

    display(
        pd.crosstab(
            validation["ScenarioID"],
            validation["Wazuh_Detected_M05delay_0s"],
            margins=True
        )
    )

# Save the combined outputs
wazuh_baseline.to_csv(
    wazuh_results_dir / "wazuh_combined_session_detections.csv",
    index=False
)

wazuh_metrics.to_csv(
    wazuh_results_dir / "wazuh_validation_metrics.csv",
    index=False
)

print("\nCombined Wazuh results saved successfully.")

In [ ]:
# Audit Wazuh detections without requiring ScenarioID

import pandas as pd

audit = pd.read_csv(
    wazuh_results_dir / "wazuh_combined_session_detections.csv"
)

audit = audit[
    audit["Partition"] == "validation"
].copy()

rule_columns = [
    "M01_Detected",
    "M02_Detected",
    "M03_Detected",
    "M04_Detected",
    "M05_Detected_0s"
]

combined_col = "Wazuh_Detected_M05delay_0s"

required = [
    "SessionID",
    "GroundTruthLabel",
    combined_col
] + rule_columns

missing = [c for c in required if c not in audit.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

for col in rule_columns + [combined_col, "GroundTruthLabel"]:
    audit[col] = pd.to_numeric(
        audit[col], errors="raise"
    ).astype(int)

assert audit["SessionID"].is_unique

assert (
    audit[rule_columns].max(axis=1)
    == audit[combined_col]
).all()

# Separate true positives and false positives
false_positives = audit[
    (audit["GroundTruthLabel"] == 0)
    & (audit[combined_col] == 1)
].copy()

true_positives = audit[
    (audit["GroundTruthLabel"] == 1)
    & (audit[combined_col] == 1)
].copy()

print("WAZUH RULE AUDIT\n")

print("Validation sessions:", len(audit))
print("True-positive sessions:", len(true_positives))
print("False-positive sessions:", len(false_positives))

# Count detections attributed to each rule
rule_audit = pd.DataFrame({
    "Rule": rule_columns,
    "TruePositiveFlags": [
        int(true_positives[c].sum())
        for c in rule_columns
    ],
    "FalsePositiveFlags": [
        int(false_positives[c].sum())
        for c in rule_columns
    ]
})

display(rule_audit)

# Identify sessions triggering multiple rules
audit["RulesTriggered"] = audit[rule_columns].sum(axis=1)

print("\nNUMBER OF RULES TRIGGERED PER SESSION")

display(
    pd.crosstab(
        audit["GroundTruthLabel"],
        audit["RulesTriggered"]
    )
)

print("\nFALSE POSITIVE SESSION DETAILS")

display(
    false_positives[
        ["SessionID"] + rule_columns
    ].head(20)
)

# Optional scenario breakdown if metadata exists
if "ScenarioID" in audit.columns:
    print("\nDETECTION BY SCENARIO")

    display(
        audit.groupby(
            ["ScenarioID", "GroundTruthLabel"]
        )[rule_columns + [combined_col]]
        .sum()
    )
else:
    print(
        "\nScenarioID is unavailable. "
        "A scenario-level audit requires the original "
        "session-to-scenario mapping."
    )

# Save results
rule_audit.to_csv(
    wazuh_results_dir / "wazuh_rule_audit.csv",
    index=False
)

false_positives.to_csv(
    wazuh_results_dir / "wazuh_false_positive_analysis.csv",
    index=False
)

print("\nWazuh audit saved successfully.")

In [ ]:
# Find the original SessionID-to-ScenarioID mapping

from pathlib import Path
import pandas as pd

search_dirs = [
    PROJECT_ROOT / "data/ground_truth",
    PROJECT_ROOT / "data/processed",
    PROJECT_ROOT / "data/raw",
]

print("FILES CONTAINING SESSION AND SCENARIO METADATA\n")

for directory in search_dirs:
    if not directory.exists():
        continue

    for path in directory.rglob("*.csv"):
        try:
            columns = pd.read_csv(path, nrows=0).columns.tolist()

            session_cols = [
                c for c in columns
                if c.lower().replace("_", "") == "sessionid"
            ]

            scenario_cols = [
                c for c in columns
                if "scenario" in c.lower()
            ]

            if session_cols and scenario_cols:
                print("File:", path.relative_to(PROJECT_ROOT))
                print("Session column:", session_cols)
                print("Scenario columns:", scenario_cols)
                print()

        except Exception:
            continue

In [ ]:
# Identify the scenarios responsible for Wazuh false positives

import pandas as pd

# Load the combined Wazuh detections
audit = pd.read_csv(
    wazuh_results_dir / "wazuh_combined_session_detections.csv",
    dtype={"SessionID": str}
)

# Load the original scenario mapping
manifest = pd.read_csv(
    PROJECT_ROOT / "data/processed/split_manifest.csv",
    dtype={"SessionID": str, "ScenarioID": str}
)

scenario_mapping = (
    manifest[["SessionID", "ScenarioID"]]
    .dropna(subset=["SessionID", "ScenarioID"])
    .drop_duplicates()
)

# Verify that each session belongs to only one scenario
if scenario_mapping["SessionID"].duplicated().any():
    raise ValueError(
        "Conflicting scenario mappings found for some sessions"
    )

if audit["SessionID"].duplicated().any():
    raise ValueError("Duplicate sessions in Wazuh results")

# Add the scenario IDs
audit = audit.drop(columns=["ScenarioID"], errors="ignore")

audit = audit.merge(
    scenario_mapping,
    on="SessionID",
    how="left",
    validate="one_to_one"
)

# Verify that every development session was matched
missing = audit["ScenarioID"].isna().sum()

print("Sessions without scenario mapping:", missing)

if missing:
    raise ValueError(
        "Some sessions could not be matched to a scenario"
    )

# Evaluate validation sessions only
validation_audit = audit[
    audit["Partition"] == "validation"
].copy()

rule_columns = [
    "M01_Detected",
    "M02_Detected",
    "M03_Detected",
    "M04_Detected",
    "M05_Detected_0s"
]

combined_col = "Wazuh_Detected_M05delay_0s"

# Convert numeric columns
for col in rule_columns + [combined_col, "GroundTruthLabel"]:
    validation_audit[col] = pd.to_numeric(
        validation_audit[col],
        errors="raise"
    ).astype(int)

# Detection performance by original simulation scenario
scenario_results = (
    validation_audit
    .groupby(["ScenarioID", "GroundTruthLabel"])
    .agg(
        TotalSessions=("SessionID", "nunique"),
        Flagged=(combined_col, "sum"),
        M01=("M01_Detected", "sum"),
        M02=("M02_Detected", "sum"),
        M03=("M03_Detected", "sum"),
        M04=("M04_Detected", "sum"),
        M05=("M05_Detected_0s", "sum")
    )
    .reset_index()
)

scenario_results["FlagRate"] = (
    scenario_results["Flagged"]
    / scenario_results["TotalSessions"]
).round(4)

print("WAZUH DETECTION PERFORMANCE BY SCENARIO")
display(scenario_results)

# Investigate false positives
false_positives = validation_audit[
    (validation_audit["GroundTruthLabel"] == 0)
    & (validation_audit[combined_col] == 1)
].copy()

print("\nFALSE POSITIVES BY BENIGN SCENARIO")

fp_summary = (
    false_positives
    .groupby("ScenarioID")
    .agg(
        FalsePositiveSessions=("SessionID", "nunique"),
        M01=("M01_Detected", "sum"),
        M02=("M02_Detected", "sum"),
        M03=("M03_Detected", "sum"),
        M04=("M04_Detected", "sum"),
        M05=("M05_Detected_0s", "sum")
    )
    .reset_index()
    .sort_values("FalsePositiveSessions", ascending=False)
)

display(fp_summary)

print("\nTotal false-positive sessions:", len(false_positives))

# Save scenario-level audit results
scenario_results.to_csv(
    wazuh_results_dir / "wazuh_detection_by_scenario.csv",
    index=False
)

fp_summary.to_csv(
    wazuh_results_dir / "wazuh_false_positives_by_scenario.csv",
    index=False
)

print("\nScenario-level audit saved successfully.")

In [ ]:
# Investigate why M01 flags N07 approved transfers

n07_ids = set(
    validation_audit.loc[
        (validation_audit["ScenarioID"] == "N07")
        & (validation_audit["M01_Detected"] == 1),
        "SessionID"
    ]
)

n07_m01_events = m01_denials[
    m01_denials["SessionID"].isin(n07_ids)
].copy()

print("N07 sessions flagged by M01:", len(n07_ids))
print("Associated denied-access events:", len(n07_m01_events))

print("\nShares involved:")
print(n07_m01_events["ShareName"].value_counts(dropna=False))

print("\nUsers involved:")
print(n07_m01_events["ResolvedUsername"].value_counts())

display(
    n07_m01_events[
        [
            "SessionID",
            "ResolvedUsername",
            "ShareName",
            "SeverityValue",
            "EventTimeUTC",
            "AvailableTimeUTC"
        ]
    ].head(15)
)